# Export Route Messages

Loads today's `sqs_routes_messages.json` and exports selected routes into separate JSON files under today's `messages/` folder.

In [ ]:
import json
from datetime import date
from pathlib import Path

PROJECT_DIR = Path.cwd()

ROUTE_EXPORTS = {
    "recorded_conversations_list.json": [
        "/v1/recordings/recorded-conversations",
        "v1/recordings/recorded-conversations",
    ],
    "ml_inference_results_list.json": ["/ml-inference-results"],
    "process_callback_list.json": ["v1/recordings/process-callback"],
}

def ordinal(day: int) -> str:
    if 11 <= day % 100 <= 13:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(day % 10, "th")
    return f"{day}{suffix}"

today = date.today()
folder_name = f"{ordinal(today.day)} {today.strftime('%B %Y')}"
target_folder = PROJECT_DIR / folder_name
route_file = target_folder / "sqs_routes_messages.json"

if not route_file.exists():
    raise FileNotFoundError(f"Route messages file not found: {route_file}")

target_folder, route_file


In [ ]:
with route_file.open("r", encoding="utf-8") as file:
    sqs_routes_messages = json.load(file)

route_counts = {
    route: len(messages) for route, messages in sqs_routes_messages.items()
}
sorted(route_counts.items(), key=lambda item: item[1], reverse=True)[:20]


In [ ]:
output_folder = target_folder / "messages"
output_folder.mkdir(parents=True, exist_ok=True)

exported_counts = {}
for file_name, routes in ROUTE_EXPORTS.items():
    messages = []
    for route in routes:
        messages.extend(sqs_routes_messages.get(route, []))
    output_file = output_folder / file_name
    with output_file.open("w", encoding="utf-8") as file:
        json.dump(messages, file, indent=2, ensure_ascii=False)
    exported_counts[file_name] = len(messages)

exported_counts
